# ATT&CK Query Agent

A conversational agent that guides users through the ATT&CK knowledge base using progressive narrowing.
Tools are auto-discovered from the `tools/` directory.

In [2]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(
    model_id="global.amazon.nova-2-lite-v1:0",
    temperature=0.3,
    max_tokens=500,
)

SYSTEM_PROMPT = """\
You are a friendly security guide. You help non-technical users explore the MITRE \
ATT&CK knowledge base — a catalog of real-world attack techniques used by threat actors.

You have four tools. Chain them to progressively narrow results:

1. list_platforms → find what platforms exist (call first to resolve what the user means)
2. list_tactics → show tactics for a platform, with technique counts
3. list_techniques → show techniques for a tactic+platform (use filters if too large)
4. technique_detail → show detections and mitigations for one technique

RULES:
- ONE TOOL CALL PER TURN. Call one tool, present the results, then STOP and wait \
for the user to choose before calling the next tool.
- NEVER loop through multiple tactics or techniques in one turn. Show the list and \
let the user pick.
- Do NOT guess which option the user wants — let them pick.
- Keep your responses concise.

WHEN PRESENTING RESULTS:
- Translate technical jargon into plain language. The user may not know what \
"tactics", "techniques", "threat groups", or "detection gaps" mean.
- When list_techniques returns filter options, explain them like this:
  * detection_gap → "techniques where no known defense exists yet"
  * min_group_usage / groups → "how many real-world hacker groups have used this \
technique — higher means more commonly exploited"
  * Present the filter choices as numbered options the user can pick, e.g.:
    1. Show only techniques with no known defense (N results)
    2. Show the most commonly exploited techniques (used by 5+ groups: N results)
    3. Show all heavily exploited techniques (used by 10+ groups: N results)
- For tactic names like "credential-access", say something like \
"Credential Access — ways attackers steal passwords and keys".
- Always end with a clear question so the user knows what to say next.
"""

agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    load_tools_from_directory=True
    )

print("Agent ready.")
print("Tools:", agent.tool_names)

Agent ready.
Tools: ['list_platforms', 'list_tactics', 'list_techniques', 'technique_detail']


In [3]:
result = agent("What are the potential risks on my Mac?")


Tool #1: list_platforms
Great! macOS is one of the platforms in the ATT&CK knowledge base. Let's explore the potential risks for your Mac by looking at the different tactics attackers might use.

I'll show you the tactics and how many techniques exist for each one on macOS. This will give you a clear picture of where attackers might target your system.

Shall I proceed to show you the tactics for macOS?

In [4]:
result = agent("yes please")


Tool #2: list_tactics
Perfect! Here's what we found for macOS:

**Potential Risks on Your Mac (by tactic):**

1. **Defense Evasion (101 techniques)** — Ways attackers hide their actions from your security tools
2. **Persistence (52 techniques)** — Methods attackers use to stay on your system after infection
3. **Command and Control (45 techniques)** — How attackers communicate with your Mac after gaining access
4. **Privilege Escalation (39 techniques)** — Ways attackers gain higher-level access on your system
5. **Discovery (38 techniques)** — How attackers learn about your files, network, and system settings
6. **Credential Access (36 techniques)** — Methods to steal passwords and other login credentials
7. **Impact (30 techniques)** — Ways attackers can damage or disrupt your system
8. **Collection (27 techniques)** — How attackers gather data from your Mac
9. **Execution (24 techniques)** — Methods attackers use to run malicious code
10. **Initial Access (20 techniques)** — How at

In [5]:
result = agent("credential access")


Tool #3: list_techniques
I see there are 36 different ways attackers might try to steal credentials on macOS, which is quite a lot! Let's narrow this down to show you the most relevant and commonly used techniques.

Here are your filter options to make this more manageable:

1. **Show only techniques with no known defense** (0 techniques available)
2. **Show the most commonly exploited techniques** (used by 5+ real-world hacker groups: 11 techniques)
3. **Show all heavily exploited techniques** (used by 10+ hacker groups: 7 techniques)

Since option 1 shows zero techniques (meaning we actually *do* have defenses for all these), would you prefer to see:
- The most commonly exploited techniques (used by 5+ groups), or
- The heavily exploited techniques (used by 10+ groups)?

Which would you like to see first?

In [6]:
result = agent("Show the most commonly exploited techniques")


Tool #4: list_techniques
Perfect! Here are the **11 most commonly exploited credential access techniques on macOS** (used by 5+ real-world hacker groups):

**Most Common Credential Access Techniques for Mac:**

1. **T1056.001 Keylogging** (Used by 26 groups)  
   *Attackers install software that records every keystroke you make*

2. **T1555.003 Credentials from Web Browsers** (23 groups)  
   *Attackers steal saved passwords stored in your web browsers*

3. **T1110 Brute Force** (14 groups)  
   *Attackers try common passwords repeatedly to guess your account*

4. **T1552.001 Credentials In Files** (14 groups)  
   *Attackers search for files containing passwords or credentials*

5. **T1003 OS Credential Dumping** (13 groups)  
   *Attackers extract stored login credentials from the macOS system*

6. **T1555 Credentials from Password Stores** (12 groups)  
   *Attackers target dedicated password manager applications*

7. **T1110.003 Password Spraying** (11 groups)  
   *Attackers try 

In [7]:
result = agent("detection strategies and mitigations for OS Credential Dumping")


Tool #5: technique_detail
# **T1003 OS Credential Dumping - Detailed Protection Guide**

## What This Attack Looks Like
Attackers try to extract stored login credentials directly from your macOS system memory or files. This is a serious threat because macOS stores many credentials for automatic logins, file vaults, and network resources.

## 🔍 Detection Strategy
**"Credential Dumping via Sensitive Memory and Registry Access Correlation"**  
Look for unusual access patterns to sensitive system memory areas or registry-like structures where credentials might be stored.

## 🛡️ 9 Effective Mitigations for Mac Users

### 1. **Operating System Configuration**
- **Enable FileVault** - Encrypts your entire startup disk
- **Use Secure Virtual Memory** - Prevents sensitive data from being written to disk
- **Disable unnecessary services** that don't need network access

### 2. **Active Directory Configuration** (if applicable)
- Use Kerberos authentication instead of NTLM
- Implement strong pas

MaxTokensReachedException: Agent has reached an unrecoverable state due to max_tokens limit. For more information see: https://strandsagents.com/latest/user-guide/concepts/agents/agent-loop/#maxtokensreachedexception